In [ ]:
import json
import numpy as np
import matplotlib
# Force an interactive GUI backend if available; fallback smoothly if headless
try:
    matplotlib.use("TkAgg")
except Exception:
    pass

import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation

# 1. SETUP CORRECTED CONE-BEAM DATASET & SAVE TO JSON

def setup_cone_beam_dataset(filename="cbct_dataset.json"):
    total_views = 60          # 60 projection angles across 360 degrees
    detector_rows = 16        # v-axis (Z-height)
    detector_cols = 16        # u-axis (horizontal channels)
    
    # Scanner Geometry (mm / units)
    D_SO = 40.0   # Source-to-Origin distance
    D_OD = 40.0   # Origin-to-Detector distance
    D_SD = D_SO + D_OD  # Source-to-Detector distance (80.0)
    pixel_size = 1.5

    angles_list = [float(i * (360.0 / total_views)) for i in range(total_views)]

    # Ground-truth 3D Phantom: Central solid sphere
    Nz, Ny, Nx = 16, 16, 16
    cz = (Nz - 1) / 2.0
    cy = (Ny - 1) / 2.0
    cx = (Nx - 1) / 2.0
    radius = 5.0

    u_coords = (np.arange(detector_cols) - (detector_cols - 1) / 2.0) * pixel_size
    v_coords = (np.arange(detector_rows) - (detector_rows - 1) / 2.0) * pixel_size

    projections = np.zeros((total_views, detector_rows, detector_cols), dtype=np.float64)

    # Analytical ray intersection with sphere of radius R
    for a_idx, deg in enumerate(angles_list):
        rad = np.deg2rad(deg)
        cos_a, sin_a = np.cos(rad), np.sin(rad)

        # Source position in world coordinates
        src = np.array([D_SO * cos_a, D_SO * sin_a, 0.0])

        for r_idx, v in enumerate(v_coords):
            for c_idx, u in enumerate(u_coords):
                # Detector pixel in world coordinates
                det = np.array([
                    -D_OD * cos_a - u * sin_a,
                    -D_OD * sin_a + u * cos_a,
                    v
                ])

                # Ray unit vector
                ray_dir = det - src
                ray_len = np.linalg.norm(ray_dir)
                d = ray_dir / ray_len

                # Intersection with centered sphere: ||src + t*d||^2 <= R^2
                # t^2 + 2*(src . d)*t + (src . src - R^2) = 0
                b = 2.0 * np.dot(src, d)
                c_term = np.dot(src, src) - radius**2
                discriminant = b**2 - 4.0 * c_term

                if discriminant > 0:
                    t1 = (-b - np.sqrt(discriminant)) / 2.0
                    t2 = (-b + np.sqrt(discriminant)) / 2.0
                    # Path length through the sphere
                    projections[a_idx, r_idx, c_idx] = max(0.0, t2 - t1)

    payload = {
        "ct_config": {
            "geometry": "cone-beam",
            "total_views": total_views,
            "detector_rows": detector_rows,
            "detector_channels": detector_cols,
            "source_to_origin_dist": D_SO,
            "source_to_detector_dist": D_SD,
            "origin_to_detector_dist": D_OD,
            "pixel_size": pixel_size
        },
        "angles": angles_list,
        "sinogram": projections.tolist()
    }

    with open(filename, "w") as f:
        json.dump(payload, f, indent=2)

    print(f"Dataset generated: {filename}")

# 2. CONE-BEAM RECONSTRUCTION OPERATORS

def forward_project_cbct(volume, angles, config):
    """Ray-integrator through estimated voxel volume."""
    Nz, Ny, Nx = volume.shape
    num_views = len(angles)
    rows = config["detector_rows"]
    cols = config["detector_channels"]
    D_SO = config["source_to_origin_dist"]
    D_OD = config["origin_to_detector_dist"]
    pixel_size = config["pixel_size"]

    u_coords = (np.arange(cols) - (cols - 1) / 2.0) * pixel_size
    v_coords = (np.arange(rows) - (rows - 1) / 2.0) * pixel_size

    cx, cy, cz = (Nx - 1) / 2.0, (Ny - 1) / 2.0, (Nz - 1) / 2.0
    projections = np.zeros((num_views, rows, cols), dtype=np.float64)

    n_samples = 32
    t_vals = np.linspace(D_SO - 7.0, D_SO + 7.0, n_samples)
    dt = 14.0 / n_samples

    for a, deg in enumerate(angles):
        rad = np.deg2rad(deg)
        cos_a, sin_a = np.cos(rad), np.sin(rad)
        src = np.array([D_SO * cos_a, D_SO * sin_a, 0.0])

        for r_idx, v in enumerate(v_coords):
            for c_idx, u in enumerate(u_coords):
                det = np.array([
                    -D_OD * cos_a - u * sin_a,
                    -D_OD * sin_a + u * cos_a,
                    v
                ])

                d = (det - src)
                d /= np.linalg.norm(d)

                val = 0.0
                for t in t_vals:
                    p = src + d * t
                    ix = int(round(p[0] + cx))
                    iy = int(round(p[1] + cy))
                    iz = int(round(p[2] + cz))

                    if 0 <= ix < Nx and 0 <= iy < Ny and 0 <= iz < Nz:
                        val += volume[iz, iy, ix] * dt

                projections[a, r_idx, c_idx] = val

    return projections


def back_project_cbct(residual_sinogram, angles, volume_shape, config):
    """Voxel-driven Feldkamp-style Cone-Beam Backprojector."""
    Nz, Ny, Nx = volume_shape
    num_views = len(angles)
    rows = config["detector_rows"]
    cols = config["detector_channels"]
    D_SO = config["source_to_origin_dist"]
    D_SD = config["source_to_detector_dist"]
    pixel_size = config["pixel_size"]

    correction = np.zeros(volume_shape, dtype=np.float64)
    cx, cy, cz = (Nx - 1) / 2.0, (Ny - 1) / 2.0, (Nz - 1) / 2.0

    det_mid_u = (cols - 1) / 2.0
    det_mid_v = (rows - 1) / 2.0

    for a, deg in enumerate(angles):
        rad = np.deg2rad(deg)
        cos_a, sin_a = np.cos(rad), np.sin(rad)

        for z in range(Nz):
            zw = z - cz
            for y in range(Ny):
                yw = y - cy
                for x in range(Nx):
                    xw = x - cx

                    # Rotate into gantry coordinates
                    depth = D_SO - (xw * cos_a + yw * sin_a)

                    if depth <= 1.0:
                        continue

                    # Transverse displacement across detector
                    s_u = -xw * sin_a + yw * cos_a

                    # Perspective projection
                    u = (D_SD / depth) * s_u
                    v = (D_SD / depth) * zw

                    c = u / pixel_size + det_mid_u
                    r = v / pixel_size + det_mid_v

                    ir = int(round(r))
                    ic = int(round(c))

                    if 0 <= ir < rows and 0 <= ic < cols:
                        weight = (D_SO / depth)**2
                        correction[z, y, x] += residual_sinogram[a, ir, ic] * weight

    correction /= num_views
    return correction

# 3. ITERATIVE RECONSTRUCTION LOOP

def iterative_reconstruction_cbct(measured, angles, volume_shape, config, iterations=8, relaxation=0.08):
    reconstruction = np.zeros(volume_shape, dtype=np.float64)

    print("\nStarting Cone-Beam Iterative Reconstruction...")
    for it in range(iterations):
        estimated = forward_project_cbct(reconstruction, angles, config)
        error = measured - estimated
        mean_err = np.mean(np.abs(error))
        print(f"Iteration {it + 1:02d}/{iterations:02d} | Mean Projection Error: {mean_err:.6f}")

        correction = back_project_cbct(error, angles, volume_shape, config)
        reconstruction += relaxation * correction
        reconstruction[reconstruction < 0] = 0.0

    return reconstruction

# 4. EXECUTION AND ROTATING 3D VISUALIZATION

if __name__ == "__main__":
    setup_cone_beam_dataset("cbct_dataset.json")

    with open("cbct_dataset.json", "r") as f:
        data = json.load(f)

    ct_config = data["ct_config"]
    angles = np.array(data["angles"], dtype=np.float64)
    sinogram = np.array(data["sinogram"], dtype=np.float64)

    Nz = ct_config["detector_rows"]
    Ny = ct_config["detector_channels"]
    Nx = ct_config["detector_channels"]
    volume_shape = (Nz, Ny, Nx)

    reconstructed_vol = iterative_reconstruction_cbct(
        measured=sinogram,
        angles=angles,
        volume_shape=volume_shape,
        config=ct_config,
        iterations=8,
        relaxation=0.08
    )

    if np.max(reconstructed_vol) > 0:
        reconstructed_vol /= np.max(reconstructed_vol)

    # 5. VISUALIZATION WITH CONTINUOUS 3D ROTATION
    fig = plt.figure(figsize=(18, 5))

    # View 1: 2D Flat Detector Projection
    ax1 = fig.add_subplot(1, 4, 1)
    im1 = ax1.imshow(sinogram[0], cmap="gray")
    ax1.set_title("Flat Projection (View 0)")
    ax1.set_xlabel("Detector Channels (u)")
    ax1.set_ylabel("Detector Rows (v)")
    plt.colorbar(im1, ax=ax1, fraction=0.046, pad=0.04)

    # View 2: Axial Slice (Z-Center)
    mid_z = Nz // 2
    ax2 = fig.add_subplot(1, 4, 2)
    im2 = ax2.imshow(reconstructed_vol[mid_z, :, :], cmap="gray", vmin=0.0, vmax=1.0)
    ax2.set_title(f"Axial Slice (Z={mid_z})")
    ax2.set_xlabel("X")
    ax2.set_ylabel("Y")
    plt.colorbar(im2, ax=ax2, fraction=0.046, pad=0.04)

    # View 3: 3D Cone-Beam Orbit Geometry
    ax3 = fig.add_subplot(1, 4, 3, projection="3d")
    D_SO = ct_config["source_to_origin_dist"]
    D_OD = ct_config["origin_to_detector_dist"]

    theta_orbit = np.linspace(0, 2 * np.pi, 100)
    ax3.plot(D_SO * np.cos(theta_orbit), D_SO * np.sin(theta_orbit), np.zeros_like(theta_orbit), 'r--', label="Source Trajectory")

    src_0 = np.array([D_SO, 0.0, 0.0])
    ax3.scatter([src_0[0]], [src_0[1]], [src_0[2]], color="red", s=60, label="X-Ray Source")

    det_u = np.linspace(-12, 12, 4)
    det_v = np.linspace(-12, 12, 4)
    U, V = np.meshgrid(det_u, det_v)
    X_det = np.full_like(U, -D_OD)
    ax3.plot_surface(X_det, U, V, color="gray", alpha=0.35)

    for corner in [[-12, -12], [-12, 12], [12, -12], [12, 12]]:
        ax3.plot([src_0[0], -D_OD], [src_0[1], corner[0]], [src_0[2], corner[1]], 'gold', lw=1.2)

    ax3.set_title("3D Scanner Geometry")
    ax3.set_xlabel("X (mm)")
    ax3.set_ylabel("Y (mm)")
    ax3.set_zlabel("Z (mm)")
    ax3.legend(loc="upper left", fontsize=7)

    # View 4: Reconstructed 3D Voxel Render
    ax4 = fig.add_subplot(1, 4, 4, projection="3d")
    threshold = 0.35
    voxels = reconstructed_vol > threshold

    if not np.any(voxels):
        voxels = reconstructed_vol > 0.1

    ax4.voxels(voxels, facecolors="deepskyblue", edgecolor="navy", alpha=0.45)
    ax4.set_title(f"Reconstructed 3D Sphere (> {threshold})")
    ax4.set_xlabel("X")
    ax4.set_ylabel("Y")
    ax4.set_zlabel("Z")

    # Animation update function for continuous rotation
    def update_rotation(frame):
        current_azim = frame % 360
        ax3.view_init(elev=25, azim=current_azim)
        ax4.view_init(elev=25, azim=current_azim)
        return ax3, ax4

    # Keep a reference to anim in a global variable so Python does not garbage-collect it
    anim = FuncAnimation(
        fig,
        update_rotation,
        frames=np.arange(0, 360, 2),
        interval=50,
        blit=False,
        repeat=True
    )

    plt.tight_layout()
    plt.show(block=True)

Dataset generated: cbct_dataset.json

Starting Cone-Beam Iterative Reconstruction...
Iteration 01/08 | Mean Projection Error: 3.705797
Iteration 02/08 | Mean Projection Error: 1.418394
Iteration 03/08 | Mean Projection Error: 1.091820
Iteration 04/08 | Mean Projection Error: 0.870065
Iteration 05/08 | Mean Projection Error: 0.712646
Iteration 06/08 | Mean Projection Error: 0.598289
Iteration 07/08 | Mean Projection Error: 0.513751
Iteration 08/08 | Mean Projection Error: 0.453871
